# Validation et correction d'un Page Object Java contre un DOM mobile
Ce notebook charge un Page Object Java, valide ses sélecteurs contre le DOM capturé, corrige automatiquement les sélecteurs invalides et génère un rapport JSON détaillé.

## 1. Importer les bibliothèques nécessaires

In [ ]:
import json
import os
import re
import difflib
from lxml import etree
from pathlib import Path

## 2. Charger le Page Object Java et le DOM

In [ ]:
workspace_root = Path('c:/Users/m.derwich/Downloads/AGENT_IA_PFE')
java_file = workspace_root / 'dom_inspector' / 'CrashDialogPO.java'
dom_file = workspace_root / 'dom_snapshots' / 'test_orange_otv.xml'

java_file.parent.mkdir(parents=True, exist_ok=True)
java_file.write_text(
    'package dom_inspector;

'
    'import io.appium.java_client.pagefactory.AndroidFindBy;
'
    'import org.openqa.selenium.WebElement;
'
    'import org.openqa.selenium.support.FindBy;
'
    'public class CrashDialogPO {
'
    '    @AndroidFindBy(id = "android:id/alertTtle")
'
    '    private WebElement title;
'
    '    @AndroidFindBy(id = "android:id/aerr_clos")
'
    '    private WebElement closeButton;
'
    '    @AndroidFindBy(xpath = "//android.widget.Button[@resource-id=\"android:id/aerr_waait\"]")
'
    '    private WebElement waitButton;
'
    '}
'
    , encoding='utf-8'
)

java_source = java_file.read_text(encoding='utf-8')
dom_xml = dom_file.read_text(encoding='utf-8')

parser = etree.XMLParser(recover=True, remove_blank_text=True)
dom_root = etree.fromstring(dom_xml.encode('utf-8'), parser=parser)

print('Java file:', java_file)
print('DOM snapshot:', dom_file)
print('DOM root tag:', dom_root.tag, 'elements:', len(dom_root.xpath('//*')))

## 3. Valider les sélecteurs

In [ ]:
ANNOTATION_PATTERN = re.compile(r'@(AndroidFindBy|FindBy)\s*\(\s*(?P<params>.+?)\s*\)')
PARAM_PATTERN = re.compile(r'(\w+)\s*=\s*\"((?:[^\\"\\]|\\.)*)\"')

def parse_annotations(source_text):
    annotations = []
    for line_number, line in enumerate(source_text.splitlines(), start=1):
        match = ANNOTATION_PATTERN.search(line)
        if not match:
            continue
        param_text = match.group('params')
        params = dict(PARAM_PATTERN.findall(param_text))
        for name, value in params.items():
            annotations.append({
                'line': line_number,
                'annotation': match.group(1),
                'param_name': name,
                'selector_name': name,
                'selector_value': value,
                'raw_line': line.strip(),
            })
    return annotations

def validate_selector(selector_name, selector_value, root):
    if selector_name == 'xpath':
        try:
            matches = root.xpath(selector_value)
            if not isinstance(matches, list):
                matches = [matches]
            if len(matches) > 0:
                return True, 'XPath valide'
            return False, 'Aucun élément trouvé'
        except Exception as exc:
            return False, f'Erreur XPath: {exc}'
    if selector_name in ['id', 'resource-id']:
        elements = root.xpath(f'//*[@resource-id=\"{selector_value}\"]')
        return (True, 'resource-id valide') if elements else (False, 'Aucun élément trouvé')
    return False, f'Type de selecteur non supporté: {selector_name}'

annotations = parse_annotations(java_source)
results = []
for annotation in annotations:
    valid, message = validate_selector(annotation['selector_name'], annotation['selector_value'], dom_root)
    results.append({
        **annotation,
        'valid': valid,
        'message': message,
    })

print('Validation results:')
for result in results:
    print(result)

## 4. Corriger automatiquement les sélecteurs invalides

In [ ]:
def find_best_matching_resource_id(root, value):
    candidates = [e.attrib.get('resource-id') for e in root.xpath('//*[@resource-id]')]
    if not candidates:
        return None
    close = difflib.get_close_matches(value, candidates, n=1, cutoff=0.5)
    return close[0] if close else None

fixed_lines = java_source.splitlines()
corrections = []
for annotation in results:
    if annotation['valid']:
        continue
    if annotation['selector_name'] in ['id', 'resource-id']:
        best = find_best_matching_resource_id(dom_root, annotation['selector_value'])
        if best and best != annotation['selector_value']:
            old = annotation['raw_line']
            new_line = old.replace(annotation['selector_value'], best)
            fixed_lines[annotation['line'] - 1] = new_line
            corrections.append({
                'line': annotation['line'],
                'original': old,
                'fixed': new_line,
                'suggested_value': best,
            })

if corrections:
    java_file.write_text('
'.join(fixed_lines) + '
', encoding='utf-8')

print('Corrections appliquées:')
for correction in corrections:
    print(correction)

## 5. Générer un rapport JSON détaillé

In [ ]:
report = {
    'java_file': str(java_file),
    'dom_snapshot': str(dom_file),
    'validated_at': __import__('datetime').datetime.utcnow().isoformat() + 'Z',
    'issues': results,
    'corrections': corrections,
}
report_dir = workspace_root / 'dom_reports'
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / 'crash_dialog_validation_report.json'
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print('Rapport sauvegardé:', report_path)